In [7]:
import warnings
warnings.simplefilter('ignore')

%matplotlib inline
import seaborn as sns
import matplotlib.pyplot as plt

import pandas as pd
import numpy as np

from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
import plotly
import plotly.graph_objs as go

init_notebook_mode(connected=True)

from sklearn.metrics import mean_squared_error as mse
from scipy.stats import linregress
from scipy.interpolate import UnivariateSpline
from datetime import datetime, timedelta

In [8]:
depths = [0,25,50,100,250]
def df_lake(d):
    df = pd.read_csv('Final_point_1_all_depths.csv')
    df.date = pd.to_datetime(df.date)
    df = df.groupby(['date', 'depth'])[['temp']].mean()
    df = df.reset_index()
    df = df.sort_values(by='date')
    df = df[df.depth == d]
    return df

cities = {'Улан-Удэ': 30823, 'Чита': 30758, 'Иркутск': 30710, 'Большое Голоустное': 30727, 'Бабушкин': 30822}
def df_city(city):
    header = ["ind", "year", "month", "day", "q_temp", 'min', 'mean', 'max', 'rain']
    df = pd.read_csv(f'{cities[city]}.txt', sep=";", engine='python', na_values=['     '], keep_default_na=False, header=None, names=header)
    df['date'] = df.apply(lambda row: datetime(int(row['year']), int(row['month']), int(row['day'])), axis=1)
    df = df[['date', 'mean']]
    df.columns = ['date', 'temp']
    df = df.dropna()
    df = df[(df.date.dt.year >= 1948) & (df.date.dt.year <= 2023)]
    df = df.sort_values(by='date')
    return df

In [9]:
tr = []
colors = ['red', 'blue', 'black', 'green', 'yellow']
c_ind = 0

for city in ['Иркутск', 'Улан-Удэ', 'Большое Голоустное', 'Бабушкин']:
    df1 = df_city(city)
    trace = go.Scatter(
        x=df1.date,
        y=df1.temp,
        mode='markers',
        marker=dict(size=10, color=colors[c_ind]),
        name=f'{city}'
    )
    c_ind += 1
    c_ind %= len(colors)
    tr.append(trace)

for d in [0,100,250]:
    df1 = df_lake(d)
    trace = go.Scatter(
        x=df1.date,
        y=df1.temp,
        mode='markers',
        marker=dict(size=10, color=colors[c_ind]),
        name=f'Байкал, глубина={d}'
    )
    c_ind += 1
    c_ind %= len(colors)
    tr.append(trace)

fig = go.Figure(data=tr)
fig.update_layout(
    title='Температура',
    xaxis_title='date',
    yaxis_title='temperature, °C',
    xaxis_type='date',
    showlegend=True
)
plotly.offline.plot(fig, filename='Температура_глубины_и_метеостанции.html', show_link=False)

'Температура_глубины_и_метеостанции.html'